# Live Data Pipeline

**Purpose:** Pull EPAM's full financial history, clean it, run it through the trained risk model, and save a dashboard-ready JSON file.

**Inputs needed:**
- `risk_model.pkl` — trained model
- `model_meta.json` — feature list and class names

**Outputs saved to `/kaggle/working/`:**
- `epam_data.json` — full dashboard data (financials + risk scores + SHAP)
- `epam_latest.json` — latest quarter only (for the dashboard header card)

## Step 1 — Install & imports

In [ ]:
!pip install -q --upgrade yfinance
print('✅ Packages ready')

In [ ]:
import json, os, warnings
from datetime import datetime
import joblib
import numpy as np
import pandas as pd
import yfinance as yf
warnings.filterwarnings('ignore')
print('✅ Imports ready')

## Step 2 — Configuration

In [ ]:
TICKER      = 'EPAM'
MODEL_DIR   = '/kaggle/input/financial-risk-model'
MODEL_PATH  = f'{MODEL_DIR}/risk_model.pkl'
META_PATH   = f'{MODEL_DIR}/model_meta.json'
OUTPUT_DIR  = '/kaggle/working'
DATA_OUT    = f'{OUTPUT_DIR}/epam_data.json'
LATEST_OUT  = f'{OUTPUT_DIR}/epam_latest.json'

print(f'Ticker     : {TICKER}')
print(f'Model path : {MODEL_PATH}')
print(f'Output     : {OUTPUT_DIR}')

# Verify model files exist
for p in [MODEL_PATH, META_PATH]:
    exists = os.path.exists(p)
    print(f'  {os.path.basename(p)}: {chr(9989) if exists else chr(10060)} {"found" if exists else "MISSING — check dataset attached"}')

## Step 3 — Load model & feature list

In [ ]:
bundle = joblib.load(MODEL_PATH)
model  = bundle['model']
le     = bundle['label_encoder']

with open(META_PATH) as f:
    meta = json.load(f)

FEATURE_COLS = meta['feature_cols']
CLASSES      = meta['classes']

print(f'✅ Model loaded')
print(f'   Classes     : {CLASSES}')
print(f'   Features    : {FEATURE_COLS}')
print(f'   Trained at  : {meta["trained_at"]}')
print(f'   Model F1    : {meta["metrics"]["weighted_f1"]}')
print(f'   Model AUC   : {meta["metrics"]["weighted_auc"]}')

## Step 4 — Pull EPAM financials from yfinance

In [ ]:
print(f'Fetching {TICKER} financials from Yahoo Finance...')
tk = yf.Ticker(TICKER)

def safe_transpose(df):
    if df is None or df.empty:
        return pd.DataFrame()
    t = df.T.copy()
    t.index = pd.to_datetime(t.index)
    t.index.name = 'date'
    return t

income   = safe_transpose(tk.quarterly_financials)
balance  = safe_transpose(tk.quarterly_balance_sheet)
cashflow = safe_transpose(tk.quarterly_cashflow)
info     = tk.info or {}

print(f'Income statement : {income.shape}')
print(f'Balance sheet    : {balance.shape}')
print(f'Cash flow        : {cashflow.shape}')
print(f'Company name     : {info.get("longName", "N/A")}')
print(f'Sector           : {info.get("sector", "N/A")}')
print(f'Market cap       : ${info.get("marketCap", 0)/1e9:.1f}B')
print(f'Employees        : {info.get("fullTimeEmployees", "N/A"):,}')

## Step 5 — Merge & engineer features

In [ ]:
def get_col(df, *candidates):
    for c in candidates:
        if c in df.columns:
            return df[c]
    return pd.Series(0, index=df.index)

# Merge all statements on same quarterly dates
frames = [f for f in [income, balance, cashflow] if not f.empty]
base = frames[0].copy()
for f in frames[1:]:
    base = base.join(f, how='outer', rsuffix='_dup')
base = base.loc[:, ~base.columns.str.endswith('_dup')].sort_index()

df = pd.DataFrame(index=base.index)
df['date']   = base.index
df['ticker'] = TICKER

# Raw financials
df['revenue']           = get_col(base, 'Total Revenue', 'Revenue')
df['gross_profit']      = get_col(base, 'Gross Profit')
df['operating_income']  = get_col(base, 'Operating Income', 'EBIT')
df['net_income']        = get_col(base, 'Net Income')
df['interest_expense']  = get_col(base, 'Interest Expense').abs()
df['total_assets']      = get_col(base, 'Total Assets')
df['total_liabilities'] = get_col(base, 'Total Liabilities Net Minority Interest', 'Total Liab')
df['total_equity']      = get_col(base, 'Stockholders Equity', 'Total Stockholders Equity')
df['current_assets']    = get_col(base, 'Current Assets')
df['current_liabilities']= get_col(base, 'Current Liabilities')
df['total_debt']        = get_col(base, 'Total Debt', 'Long Term Debt')
df['cash']              = get_col(base, 'Cash And Cash Equivalents', 'Cash Cash Equivalents And Short Term Investments')
df['operating_cash_flow']= get_col(base, 'Operating Cash Flow', 'Cash Flow From Continuing Operating Activities')
df['capex']             = get_col(base, 'Capital Expenditure').abs()
df['free_cash_flow']    = df['operating_cash_flow'] - df['capex']

# Engineered ratios
eps = 1e-9
df['gross_margin']       = df['gross_profit']      / (df['revenue'].abs() + eps)
df['operating_margin']   = df['operating_income']  / (df['revenue'].abs() + eps)
df['net_margin']         = df['net_income']         / (df['revenue'].abs() + eps)
df['fcf_margin']         = df['free_cash_flow']     / (df['revenue'].abs() + eps)
df['roe']                = df['net_income']         / (df['total_equity'].abs() + eps)
df['roa']                = df['net_income']         / (df['total_assets'].abs() + eps)
df['debt_to_equity']     = df['total_debt']         / (df['total_equity'].abs() + eps)
df['current_ratio']      = df['current_assets']     / (df['current_liabilities'].abs() + eps)
df['interest_coverage']  = df['operating_income']   / (df['interest_expense'] + eps)
df['asset_turnover']     = df['revenue']             / (df['total_assets'].abs() + eps)
df['revenue_growth_yoy'] = df['revenue'].pct_change(periods=4)

# Clip outliers
ratio_cols = ['gross_margin','operating_margin','net_margin','fcf_margin',
              'roe','roa','debt_to_equity','current_ratio',
              'interest_coverage','asset_turnover','revenue_growth_yoy']
df[ratio_cols] = df[ratio_cols].clip(-100, 100)

# Fill NaN with median (same as training)
for col in FEATURE_COLS:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

df = df.reset_index(drop=True)
print(f'✅ Features engineered: {df.shape[0]} quarters x {df.shape[1]} columns')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
df[['date'] + ratio_cols].tail(6).round(3)

## Step 6 — Run risk model predictions

In [ ]:
X = df[FEATURE_COLS].values.astype(np.float32)

# Predict class and probabilities
y_pred   = model.predict(X)
y_proba  = model.predict_proba(X)

df['risk_label']      = le.inverse_transform(y_pred)
df['risk_score']      = y_proba.max(axis=1).round(4)  # confidence of predicted class

# Probability for each class
for i, cls in enumerate(le.classes_):
    df[f'prob_{cls}'] = y_proba[:, i].round(4)

print(f'✅ Predictions complete')
print(f'\nRisk label distribution:')
print(df['risk_label'].value_counts().to_string())
print(f'\nLatest quarter prediction:')
latest = df.sort_values('date').iloc[-1]
print(f'  Date       : {latest["date"].date()}')
print(f'  Risk label : {latest["risk_label"]}')
print(f'  Confidence : {latest["risk_score"]:.1%}')

## Step 7 — Compute SHAP explanations per quarter

In [ ]:
import shap
print('Computing SHAP explanations...')

explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

# Handle 3D output (samples, features, classes) from newer SHAP
if isinstance(shap_values, list):
    mean_shap = np.mean([np.abs(sv) for sv in shap_values], axis=0)
elif shap_values.ndim == 3:
    mean_shap = np.abs(shap_values).mean(axis=2)
else:
    mean_shap = np.abs(shap_values)

# Add top 3 risk drivers per quarter
shap_records = []
for i in range(len(df)):
    row_shap = mean_shap[i]
    top_idx  = np.argsort(row_shap)[::-1][:3]
    drivers  = [
        {'feature': FEATURE_COLS[j], 'importance': round(float(row_shap[j]), 4)}
        for j in top_idx
    ]
    shap_records.append(drivers)

df['top_risk_drivers'] = shap_records
print(f'✅ SHAP done — top 3 drivers saved per quarter')
print(f'\nLatest quarter top drivers:')
for d in df.iloc[-1]['top_risk_drivers']:
    print(f'  {d["feature"]:<25} {d["importance"]:.4f}')

## Step 8 — Add company metadata

In [ ]:
company_meta = {
    'ticker':       TICKER,
    'name':         info.get('longName', 'EPAM Systems'),
    'sector':       info.get('sector', 'Technology'),
    'industry':     info.get('industry', 'Information Technology Services'),
    'country':      info.get('country', 'United States'),
    'employees':    info.get('fullTimeEmployees', None),
    'market_cap':   info.get('marketCap', None),
    'website':      info.get('website', None),
    'description':  info.get('longBusinessSummary', '')[:300],
    'fetched_at':   datetime.now().isoformat(),
    'model_f1':     meta['metrics']['weighted_f1'],
    'model_auc':    meta['metrics']['weighted_auc'],
}

print('✅ Company metadata ready')
for k, v in company_meta.items():
    if k != 'description':
        print(f'  {k:<15} : {v}')

## Step 9 — Save dashboard JSON files

In [ ]:
# Convert DataFrame to list of records for JSON
records = []
for _, row in df.iterrows():
    record = {
        'date':             row['date'].strftime('%Y-%m-%d'),
        'ticker':           row['ticker'],
        # Raw financials (in millions for readability)
        'revenue':          round(float(row['revenue']) / 1e6, 2) if pd.notna(row['revenue']) else None,
        'gross_profit':     round(float(row['gross_profit']) / 1e6, 2) if pd.notna(row['gross_profit']) else None,
        'operating_income': round(float(row['operating_income']) / 1e6, 2) if pd.notna(row['operating_income']) else None,
        'net_income':       round(float(row['net_income']) / 1e6, 2) if pd.notna(row['net_income']) else None,
        'free_cash_flow':   round(float(row['free_cash_flow']) / 1e6, 2) if pd.notna(row['free_cash_flow']) else None,
        'total_debt':       round(float(row['total_debt']) / 1e6, 2) if pd.notna(row['total_debt']) else None,
        'cash':             round(float(row['cash']) / 1e6, 2) if pd.notna(row['cash']) else None,
        # Ratios
        'gross_margin':     round(float(row['gross_margin']), 4) if pd.notna(row['gross_margin']) else None,
        'operating_margin': round(float(row['operating_margin']), 4) if pd.notna(row['operating_margin']) else None,
        'net_margin':       round(float(row['net_margin']), 4) if pd.notna(row['net_margin']) else None,
        'current_ratio':    round(float(row['current_ratio']), 4) if pd.notna(row['current_ratio']) else None,
        'debt_to_equity':   round(float(row['debt_to_equity']), 4) if pd.notna(row['debt_to_equity']) else None,
        'roe':              round(float(row['roe']), 4) if pd.notna(row['roe']) else None,
        'roa':              round(float(row['roa']), 4) if pd.notna(row['roa']) else None,
        'interest_coverage':round(float(row['interest_coverage']), 4) if pd.notna(row['interest_coverage']) else None,
        'revenue_growth_yoy': round(float(row['revenue_growth_yoy']), 4) if pd.notna(row['revenue_growth_yoy']) else None,
        # Risk predictions
        'risk_label':       row['risk_label'],
        'risk_score':       float(row['risk_score']),
        'prob_high_risk':   float(row['prob_high_risk']),
        'prob_low_risk':    float(row['prob_low_risk']),
        'prob_medium_risk': float(row['prob_medium_risk']),
        'top_risk_drivers': row['top_risk_drivers'],
    }
    records.append(record)

# Sort by date ascending
records = sorted(records, key=lambda x: x['date'])

# Full dashboard JSON
full_output = {
    'company':  company_meta,
    'quarters': records,
    'summary': {
        'total_quarters':   len(records),
        'date_range':       f"{records[0]['date']} → {records[-1]['date']}",
        'latest_risk':      records[-1]['risk_label'],
        'latest_confidence':records[-1]['risk_score'],
        'risk_distribution':{
            'low_risk':    sum(1 for r in records if r['risk_label'] == 'low_risk'),
            'medium_risk': sum(1 for r in records if r['risk_label'] == 'medium_risk'),
            'high_risk':   sum(1 for r in records if r['risk_label'] == 'high_risk'),
        }
    }
}

with open(DATA_OUT, 'w') as f:
    json.dump(full_output, f, indent=2, default=str)
print(f'✅ Full dashboard data saved → {DATA_OUT}')

# Latest quarter only (for header card)
latest_output = {
    'company': company_meta,
    'latest':  records[-1],
    'summary': full_output['summary'],
}
with open(LATEST_OUT, 'w') as f:
    json.dump(latest_output, f, indent=2, default=str)
print(f'✅ Latest quarter saved      → {LATEST_OUT}')

# File sizes
print('\n── Output files ───────────────────────────────')
for fpath in [DATA_OUT, LATEST_OUT]:
    size = os.path.getsize(fpath)
    print(f'  {os.path.basename(fpath):<25} {size/1024:.1f} KB')

## Step 10 — Preview the output

In [ ]:
print('── Company ────────────────────────────────────')
print(f'  Name        : {full_output["company"]["name"]}')
print(f'  Market cap  : ${full_output["company"]["market_cap"]/1e9:.1f}B')
print(f'  Employees   : {full_output["company"]["employees"]:,}')

print('\n── Summary ─────────────────────────────────────')
s = full_output['summary']
print(f'  Quarters    : {s["total_quarters"]}')
print(f'  Date range  : {s["date_range"]}')
print(f'  Latest risk : {s["latest_risk"]} ({s["latest_confidence"]:.1%} confidence)')
print(f'  Distribution: {s["risk_distribution"]}')

print('\n── Latest quarter ──────────────────────────────')
lat = records[-1]
print(f'  Date              : {lat["date"]}')
print(f'  Revenue           : ${lat["revenue"]}M')
print(f'  Operating margin  : {lat["operating_margin"]:.1%}')
print(f'  Current ratio     : {lat["current_ratio"]:.2f}')
print(f'  Debt-to-equity    : {lat["debt_to_equity"]:.2f}')
print(f'  Risk label        : {lat["risk_label"]}')
print(f'  Top risk drivers  :')
for d in lat['top_risk_drivers']:
    print(f'    {d["feature"]:<25} {d["importance"]:.4f}')